TODO: move this into the right place (notebooks are just for figures as I understand it)

In [1]:
import multiprocessing
import os
import platform
from pathlib import Path

import numpy as np
import openscm_units
import pandas as pd
import pandas_indexing as pix
import pandas_openscm
import pint
from pandas_openscm.index_manipulation import update_index_levels_func

from gcages.cmip7_scenariomip.harmonisation import (
    create_cmip7_scenariomip_global_harmoniser,
    load_cmip7_scenariomip_historical_emissions,
)
from gcages.cmip7_scenariomip.infilling import (
    CMIP7ScenarioMIPInfiller,
    load_cmip7_scenariomip_ghg_inversions,
    load_cmip7_scenariomip_historical_emissions,
    load_cmip7_scenariomip_infilling_db,
)
from gcages.cmip7_scenariomip.harmonisation import load_cmip7_scenariomip_historical_emissions, load_aneris_overrides_file
from gcages.harmonisation.aneris import AnerisHarmoniser
from gcages.cmip7_scenariomip.post_processing import CMIP7ScenarioMIPPostProcessor
from gcages.cmip7_scenariomip.scm_running import CMIP7ScenarioMIPSCMRunner
from gcages.harmonisation.common import assert_harmonised
from gcages.renaming import (
    SupportedNamingConventions,
    convert_variable_name,
    rename_variables,
)
from gcages.units_helpers import strip_pint_incompatible_characters_from_units

In [2]:
pint.set_application_registry(openscm_units.unit_registry)

In [3]:
pandas_openscm.register_pandas_accessors()

In [4]:
feather_file = Path("emissions.feather")
if feather_file.exists():
    # Shortcut
    emissions = pd.read_feather(feather_file)
    
else:
    # Unclear if we can commit this file or not. Ignoring for now.
    start = pd.read_excel("SCI-2025_v1.0_pathways_ensemble_global.xlsx", sheet_name="data")
    tmp = start.copy()
    tmp.columns = tmp.columns.str.lower()
    tmp = tmp.set_index(["model", "scenario", "region", "variable", "unit"])
    emissions = tmp.loc[pix.ismatch(variable="Emissions**", region="World")]
    emissions.to_feather(feather_file)

emissions.columns = emissions.columns.astype(int)
emissions

2010  \
model             scenario           region variable              unit                            
AIM/CGE 2.0       SSP1-19            World  Emissions|BC          Mt BC/yr             7.337400   
                                            Emissions|CH4         Mt CH4/yr          368.104600   
                                            Emissions|CH4|AFOLU   Mt CH4/yr          179.530900   
                                            Emissions|CH4|Energy  Mt CH4/yr          124.394400   
                                            Emissions|CO          Mt CO/yr           870.610400   
...                                                                                         ...   
WITCH-GLOBIOM 4.4 CD-LINKS-No-Policy World  Emissions|F-Gases     Mt CO2-equiv/yr   1032.189454   
                                            Emissions|Kyoto Gases Mt CO2-equiv/yr  48676.563680   
                                            Emissions|N2O         kt N2O/yr        10235.569004   
                                            Emissions|N2O|AFOLU   kt N2O/yr         8779.675302   
                                            Emissions|Sulfur      Mt SO2/yr           81.655701   

                                                                                           2015  \
model             scenario           region variable              unit                            
AIM/CGE 2.0       SSP1-19            World  Emissions|BC          Mt BC/yr             6.661400   
                                            Emissions|CH4         Mt CH4/yr          371.281100   
                                            Emissions|CH4|AFOLU   Mt CH4/yr          184.616800   
                                            Emissions|CH4|Energy  Mt CH4/yr          120.509300   
                                            Emissions|CO          Mt CO/yr           818.449800   
...                                                                                         ...   
WITCH-GLOBIOM 4.4 CD-LINKS-No-Policy World  Emissions|F-Gases     Mt CO2-equiv/yr   1872.142705   
                                            Emissions|Kyoto Gases Mt CO2-equiv/yr  52097.101002   
                                            Emissions|N2O         kt N2O/yr        10330.054024   
                                            Emissions|N2O|AFOLU   kt N2O/yr         8850.331096   
                                            Emissions|Sulfur      Mt SO2/yr           84.329131   

                                                                                           2020  \
model             scenario           region variable              unit                            
AIM/CGE 2.0       SSP1-19            World  Emissions|BC          Mt BC/yr             5.985400   
                                            Emissions|CH4         Mt CH4/yr          374.457600   
                                            Emissions|CH4|AFOLU   Mt CH4/yr          189.702700   
                                            Emissions|CH4|Energy  Mt CH4/yr          116.624200   
                                            Emissions|CO          Mt CO/yr           766.289200   
...                                                                                         ...   
WITCH-GLOBIOM 4.4 CD-LINKS-No-Policy World  Emissions|F-Gases     Mt CO2-equiv/yr   2642.220050   
                                            Emissions|Kyoto Gases Mt CO2-equiv/yr  58277.316825   
                                            Emissions|N2O         kt N2O/yr        10596.100239   
                                            Emissions|N2O|AFOLU   kt N2O/yr         8996.243223   
                                            Emissions|Sulfur      Mt SO2/yr           82.158099   

                                                                                           2025  \
model             scenario           region variable              unit                            
AIM/CGE 2.0       SSP1-19    

## Various hacks to the input

All the usual things that are needed to deal with IAM data.

In [5]:
emissions = emissions.sort_index(axis="columns")

unusable_too_late_start = (
    emissions.isnull()
    .idxmin(axis="columns")
    .groupby(["model", "scenario"])
    .max()
    .sort_values()
)
unusable_too_late_start = unusable_too_late_start[unusable_too_late_start > 2023]
print(f"Unusuable because they start after 2023:\n{unusable_too_late_start}")
emissions_usable = emissions.loc[
    ~pandas_openscm.indexing.multi_index_match(
        emissions.index,
        unusable_too_late_start.index,
    )
]

for y in range(2010, 2100 + 1):
    if y not in emissions_usable:
        emissions_usable[y] = np.nan
        
emissions = (
    emissions_usable.T.interpolate(method="index")
    .T.sort_index(axis="columns")
    .loc[:, 2023:]
)
if emissions.isnull().any().any():
    raise AssertionError

Unusuable because they start after 2023:
model                  scenario                                                   
MESSAGEix-GLOBIOM 1.1  GENIE DACCS DACm-LP-median-stor3-final_700                     2025
                       GENIE DACCS DACl-MP-median-stor3-final_500                     2025
                       GENIE DACCS DACm-MP-median-stor3-phs-govmSSP2-CO2total_1000    2025
                       GENIE DACCS DACm-MP-median-stor3-phs-govmSSP1-CO2total_700     2025
                       GENIE DACCS DACm-MP-median-stor3-phs-govmSSP1-CO2total_1000    2025
                       GENIE DACCS DACm-MP-median-stor3-final_700                     2025
                       GENIE DACCS DACm-MP-median-stor3-final_500                     2025
                       GENIE DACCS DACm-MP-median-stor3-final_1000                    2025
                       GENIE DACCS DACm-LP-median-stor3-phs-govmSSP2-CO2total_1000    2025
                       GENIE DACCS DACm-LP-median-stor3-p

In [6]:
emissions_renamed = rename_variables(
    emissions.loc[
        pix.ismatch(
            variable=[
                "Emissions|*",
                "Emissions|HFC|**",
                # Almost certainly need some reaggregation for some scenarios
                "Emissions|CO2|Energy and Industrial Processes",
                "Emissions|CO2|AFOLU",
            ]
        )
        & ~pix.isin(
            variable=[
                "Emissions|F-Gases",
                "Emissions|PFC",
                "Emissions|HFC",
                "Emissions|Kyoto Gases",
                "Emissions|CO2",
            ]
        )
    ],
    from_convention=SupportedNamingConventions.CMIP7_SCENARIOMIP,
    to_convention=SupportedNamingConventions.GCAGES,
)
emissions_renamed = strip_pint_incompatible_characters_from_units(
    emissions_renamed, units_index_level="unit"
)

sorted(emissions_renamed.index.droplevel(["model", "scenario", "region"]).unique())

[('Emissions|BC', 'Mt BC/yr'),
 ('Emissions|C2F6', 'kt C2F6/yr'),
 ('Emissions|C6F14', 'kt C6F14/yr'),
 ('Emissions|CF4', 'kt CF4/yr'),
 ('Emissions|CH4', 'Mt CH4/yr'),
 ('Emissions|CO', 'Mt CO/yr'),
 ('Emissions|CO2|Biosphere', 'Mt CO2/yr'),
 ('Emissions|CO2|Fossil', 'Mt CO2/yr'),
 ('Emissions|HFC125', 'kt HFC125/yr'),
 ('Emissions|HFC134a', 'kt HFC134a/yr'),
 ('Emissions|HFC143a', 'kt HFC143a/yr'),
 ('Emissions|HFC227ea', 'kt HFC227ea/yr'),
 ('Emissions|HFC23', 'kt HFC23/yr'),
 ('Emissions|HFC245fa', 'kt HFC245fa/yr'),
 ('Emissions|HFC32', 'kt HFC32/yr'),
 ('Emissions|HFC4310mee', 'kt HFC4310/yr'),
 ('Emissions|N2O', 'kt N2O/yr'),
 ('Emissions|NH3', 'Mt NH3/yr'),
 ('Emissions|NMVOC', 'Mt VOC/yr'),
 ('Emissions|NOx', 'Mt NO2/yr'),
 ('Emissions|OC', 'Mt OC/yr'),
 ('Emissions|SF6', 'kt SF6/yr'),
 ('Emissions|SOx', 'Mt SO2/yr')]

In [7]:
run_all = True
# run_all = False
if run_all:
    emissions_run = emissions_renamed
    
else:
    models_included = []
    new_idx_levels = []
    for model, scenario in emissions_renamed.index.droplevel(
        ["region", "variable", "unit"]
    ).unique():
        if model in models_included:
            continue

        models_included.append(model)
        new_idx_levels.append((model, scenario))

    emissions_run = emissions_renamed.loc[
        pandas_openscm.indexing.multi_index_match(
            emissions_renamed.index,
            pd.MultiIndex.from_tuples(new_idx_levels, names=["model", "scenario"]),
        )
    ]

    # emissions_run = emissions_run.loc[pix.isin(model="REMIND 2.1", scenario="R2p1-SSP2-1100")]
    
emissions_run

2023  \
model             scenario           region variable                unit                      
AIM/CGE 2.0       SSP1-19            World  Emissions|BC            Mt BC/yr       5.429410   
                                            Emissions|CH4           Mt CH4/yr    300.867810   
                                            Emissions|CO            Mt CO/yr     723.699490   
                                            Emissions|CO2|Biosphere Mt CO2/yr   2506.979860   
                                            Emissions|CO2|Fossil    Mt CO2/yr  29273.990990   
...                                                                                     ...   
WITCH-GLOBIOM 4.4 CD-LINKS-No-Policy World  Emissions|CH4           Mt CH4/yr    404.808829   
                                            Emissions|CO2|Biosphere Mt CO2/yr   2660.334724   
                                            Emissions|CO2|Fossil    Mt CO2/yr  43016.332284   
                                            Emissions|N2O           kt N2O/yr  10704.134052   
                                            Emissions|SOx           Mt SO2/yr     81.361598   

                                                                                       2024  \
model             scenario           region variable                unit                      
AIM/CGE 2.0       SSP1-19            World  Emissions|BC            Mt BC/yr       5.244080   
                                            Emissions|CH4           Mt CH4/yr    276.337880   
                                            Emissions|CO            Mt CO/yr     709.502920   
                                            Emissions|CO2|Biosphere Mt CO2/yr   2138.037580   
                                            Emissions|CO2|Fossil    Mt CO2/yr  27825.293620   
...                                                                                     ...   
WITCH-GLOBIOM 4.4 CD-LINKS-No-Policy World  Emissions|CH4           Mt CH4/yr    408.792866   
                                            Emissions|CO2|Biosphere Mt CO2/yr   2663.847629   
                                            Emissions|CO2|Fossil    Mt CO2/yr  44019.694709   
                                            Emissions|N2O           kt N2O/yr  10740.145323   
                                            Emissions|SOx           Mt SO2/yr     81.096097   

                                                                                       2025  \
model             scenario           region variable                unit                      
AIM/CGE 2.0       SSP1-19            World  Emissions|BC            Mt BC/yr       5.058750   
                                            Emissions|CH4           Mt CH4/yr    251.807950   
                                            Emissions|CO            Mt CO/yr     695.306350   
                                            Emissions|CO2|Biosphere Mt CO2/yr   1769.095300   
                                            Emissions|CO2|Fossil    Mt CO2/yr  26376.596250   
...                                                                                     ...   
WITCH-GLOBIOM 4.4 CD-LINKS-No-Policy World  Emissions|CH4           Mt CH4/yr    412.776903   
                                            Emissions|CO2|Biosphere Mt CO2/yr   2667.360534   
                                            Emissions|CO2|Fossil    Mt CO2/yr  45023.057135   
                                            Emissions|N2O           kt N2O/yr  10776.156593   
                                            Emissions|SOx           Mt SO2/yr     80.830597   

                                                                                       2026  \
model             scenario           region variable                unit                      
AIM/CGE 2.0       SSP1-19            World  Emissions|BC            Mt BC/yr       4.873420   
                                            Emissions|CH4           Mt CH4/yr    227.278020   

In [8]:
# Round to get rid of spurious negatives
emissions_run.loc[
    pix.ismatch(variable="**HFC**")
    & (emissions_run.round(3) == 0.0).any(axis="columns")
] = 0.0
neg_vals = (emissions_run.loc[~pix.ismatch(variable="**CO2**")] < 0).any(axis=1)
unusable_because_of_neg = (
    neg_vals[neg_vals].index.droplevel(["region", "variable", "unit"]).drop_duplicates()
)
print(
    f"Unusuable because of inexplicable negatives:\n{emissions_run.loc[~pix.ismatch(variable='**CO2**')].loc[neg_vals].min(axis=1)}"
)

emissions_run = emissions_run.loc[
    ~pandas_openscm.indexing.multi_index_match(
        emissions_run.index, unusable_because_of_neg
    )
]
emissions_run


Unusuable because of inexplicable negatives:
model                          scenario                     region  variable           unit         
MESSAGEix-GLOBIOM 1.1-BMT-R12  NAVIGATE Demand-1.5°C-all_u  World   Emissions|HFC125   kt HFC125/yr    -138.128465
                                                                    Emissions|HFC143a  kt HFC143a/yr     -3.408424
                                                                    Emissions|HFC32    kt HFC32/yr     -141.337757
                               NAVIGATE Demand-1.5°C-ref    World   Emissions|HFC125   kt HFC125/yr    -138.128465
                                                                    Emissions|HFC143a  kt HFC143a/yr     -3.408424
                                                                    Emissions|HFC32    kt HFC32/yr     -141.337757
                               NAVIGATE Demand-2.0°C-act_u  World   Emissions|HFC125   kt HFC125/yr     -49.451111
                               NAVIGATE Demand-2.

2023  \
model             scenario           region variable                unit                      
AIM/CGE 2.0       SSP1-19            World  Emissions|BC            Mt BC/yr       5.429410   
                                            Emissions|CH4           Mt CH4/yr    300.867810   
                                            Emissions|CO            Mt CO/yr     723.699490   
                                            Emissions|CO2|Biosphere Mt CO2/yr   2506.979860   
                                            Emissions|CO2|Fossil    Mt CO2/yr  29273.990990   
...                                                                                     ...   
WITCH-GLOBIOM 4.4 CD-LINKS-No-Policy World  Emissions|CH4           Mt CH4/yr    404.808829   
                                            Emissions|CO2|Biosphere Mt CO2/yr   2660.334724   
                                            Emissions|CO2|Fossil    Mt CO2/yr  43016.332284   
                                            Emissions|N2O           kt N2O/yr  10704.134052   
                                            Emissions|SOx           Mt SO2/yr     81.361598   

                                                                                       2024  \
model             scenario           region variable                unit                      
AIM/CGE 2.0       SSP1-19            World  Emissions|BC            Mt BC/yr       5.244080   
                                            Emissions|CH4           Mt CH4/yr    276.337880   
                                            Emissions|CO            Mt CO/yr     709.502920   
                                            Emissions|CO2|Biosphere Mt CO2/yr   2138.037580   
                                            Emissions|CO2|Fossil    Mt CO2/yr  27825.293620   
...                                                                                     ...   
WITCH-GLOBIOM 4.4 CD-LINKS-No-Policy World  Emissions|CH4           Mt CH4/yr    408.792866   
                                            Emissions|CO2|Biosphere Mt CO2/yr   2663.847629   
                                            Emissions|CO2|Fossil    Mt CO2/yr  44019.694709   
                                            Emissions|N2O           kt N2O/yr  10740.145323   
                                            Emissions|SOx           Mt SO2/yr     81.096097   

                                                                                       2025  \
model             scenario           region variable                unit                      
AIM/CGE 2.0       SSP1-19            World  Emissions|BC            Mt BC/yr       5.058750   
                                            Emissions|CH4           Mt CH4/yr    251.807950   
                                            Emissions|CO            Mt CO/yr     695.306350   
                                            Emissions|CO2|Biosphere Mt CO2/yr   1769.095300   
                                            Emissions|CO2|Fossil    Mt CO2/yr  26376.596250   
...                                                                                     ...   
WITCH-GLOBIOM 4.4 CD-LINKS-No-Policy World  Emissions|CH4           Mt CH4/yr    412.776903   
                                            Emissions|CO2|Biosphere Mt CO2/yr   2667.360534   
                                            Emissions|CO2|Fossil    Mt CO2/yr  45023.057135   
                                            Emissions|N2O           kt N2O/yr  10776.156593   
                                            Emissions|SOx           Mt SO2/yr     80.830597   

                                                                                       2026  \
model             scenario           region variable                unit                      
AIM/CGE 2.0       SSP1-19            World  Emissions|BC            Mt BC/yr       4.873420   
                                            Emissions|CH4           Mt CH4/yr    227.278020   

In [9]:
if emissions_run.isnull().any().any():
    raise AssertionError

## Harmonisation

In [10]:
harmonisation_year = 2023
run_checks = True
progress = True
# Sorry Ben, you can adjust for slurm
n_processes = multiprocessing.cpu_count()
# n_processes = None

historical_emissions_file = Path(
    "history_cmip7_scenariomip.csv"
)
aneris_overrides_file = Path(
    "aneris-overrides-global.csv"
)

historical_emissions = load_cmip7_scenariomip_historical_emissions(
    filepath=historical_emissions_file,
    # check_hash=True,
)

# Drop out the model and scenario levels
historical_emissions = historical_emissions.reset_index(
    historical_emissions.index.names.difference(["variable", "region", "unit"]),  # type: ignore # pandas-stubs out of date
    drop=True,
)

# Use gcages naming to match pre-processed outputs.
historical_emissions = update_index_levels_func(
    historical_emissions,
    {
        "variable": lambda x: convert_variable_name(
            x,
            from_convention=SupportedNamingConventions.CMIP7_SCENARIOMIP,
            to_convention=SupportedNamingConventions.GCAGES,
        )
    },
    copy=False,
)

aneris_overrides = load_aneris_overrides_file(aneris_overrides_file)
# Type juggling for mypy: from series to dataframe back to series
# TODO: remove this as it isn't needed for pandas-openscm 0.8.1
aneris_overrides_df = aneris_overrides.to_frame(name="method")
updated_df = update_index_levels_func(
    aneris_overrides_df,
    {
        "variable": lambda x: convert_variable_name(
            x,
            from_convention=SupportedNamingConventions.CMIP7_SCENARIOMIP,
            to_convention=SupportedNamingConventions.GCAGES,
        )
    },
    copy=False,
)
aneris_overrides = updated_df["method"]

harmoniser = AnerisHarmoniser(
    historical_emissions=historical_emissions,
    harmonisation_year=harmonisation_year,
    aneris_overrides=aneris_overrides,
    run_checks=run_checks,
    progress=progress,
    n_processes=n_processes,
)

In [11]:
harmonised_global = harmoniser(emissions_run)
harmonised_global

0it [00:00, ?it/s]

Scenarios to harmonise:   0%|          | 0/1563 [00:00<?, ?it/s]

INFO:root:Harmonizing with reduce_ratio_2080
INFO:root:Harmonizing with reduce_ratio_2080
INFO:root:Harmonizing with reduce_ratio_2080
INFO:root:Harmonizing with reduce_ratio_2080
INFO:root:Harmonizing with reduce_offset_2150_cov
INFO:root:Harmonizing with reduce_ratio_2080
INFO:root:Harmonizing with reduce_offset_2150_cov
INFO:root:Harmonizing with reduce_offset_2150_cov
INFO:root:Harmonizing with constant_ratio
INFO:root:Harmonizing with constant_ratio
INFO:root:Harmonizing with reduce_offset_2150_cov
INFO:root:Harmonizing with reduce_offset_2150_cov
INFO:root:Harmonizing with constant_ratio
INFO:root:Harmonizing with model_zero
INFO:root:Harmonizing with reduce_ratio_2080
INFO:root:Harmonizing with constant_ratio
INFO:root:Harmonizing with model_zero
INFO:root:Harmonizing with constant_ratio
INFO:root:Harmonizing with model_zero
INFO:root:Harmonizing with model_zero
INFO:root:Harmonizing with reduce_offset_2150_cov
INFO:root:Harmonizing with reduce_ratio_2080
INFO:root:Harmonizing w

2023  \
model             scenario           region variable                unit                      
AIM/CGE 2.0       SSP1-19            World  Emissions|BC            Mt BC/yr       7.219245   
                                            Emissions|CH4           Mt CH4/yr    363.489901   
                                            Emissions|CO            Mt CO/yr     774.053434   
                                            Emissions|CO2|Biosphere Mt CO2/yr   3627.550667   
                                            Emissions|CO2|Fossil    Mt CO2/yr  38712.217040   
...                                                                                     ...   
WITCH-GLOBIOM 4.4 CD-LINKS-No-Policy World  Emissions|CH4           Mt CH4/yr    363.489901   
                                            Emissions|CO2|Biosphere Mt CO2/yr   3627.550667   
                                            Emissions|CO2|Fossil    Mt CO2/yr  38712.217040   
                                            Emissions|N2O           kt N2O/yr  11289.003671   
                                            Emissions|SOx           Mt SO2/yr     71.902038   

                                                                                       2024  \
model             scenario           region variable                unit                      
AIM/CGE 2.0       SSP1-19            World  Emissions|BC            Mt BC/yr       6.942491   
                                            Emissions|CH4           Mt CH4/yr    332.845294   
                                            Emissions|CO            Mt CO/yr     758.003015   
                                            Emissions|CO2|Biosphere Mt CO2/yr   3249.784994   
                                            Emissions|CO2|Fossil    Mt CO2/yr  36639.056595   
...                                                                                     ...   
WITCH-GLOBIOM 4.4 CD-LINKS-No-Policy World  Emissions|CH4           Mt CH4/yr    367.799315   
                                            Emissions|CO2|Biosphere Mt CO2/yr   3623.447698   
                                            Emissions|CO2|Fossil    Mt CO2/yr  39692.457424   
                                            Emissions|N2O           kt N2O/yr  11316.687192   
                                            Emissions|SOx           Mt SO2/yr     71.832822   

                                                                                       2025  \
model             scenario           region variable                unit                      
AIM/CGE 2.0       SSP1-19            World  Emissions|BC            Mt BC/yr       6.667881   
                                            Emissions|CH4           Mt CH4/yr    302.379832   
                                            Emissions|CO            Mt CO/yr     741.987254   
                                            Emissions|CO2|Biosphere Mt CO2/yr   2872.019322   
                                            Emissions|CO2|Fossil    Mt CO2/yr  34582.284727   
...                                                                                     ...   
WITCH-GLOBIOM 4.4 CD-LINKS-No-Policy World  Emissions|CH4           Mt CH4/yr    372.122996   
                                            Emissions|CO2|Biosphere Mt CO2/yr   3619.344730   
                                            Emissions|CO2|Fossil    Mt CO2/yr  40676.220409   
                                            Emissions|N2O           kt N2O/yr  11344.301674   
                                            Emissions|SOx           Mt SO2/yr     71.762522   

                                                                                       2026  \
model             scenario           region variable                unit                      
AIM/CGE 2.0       SSP1-19            World  Emissions|BC            Mt BC/yr       6.395415   
                                            Emissions|CH4           Mt CH4/yr    272.093514   

## Infilling

In [14]:
# TODO: discuss - we probably want to use a better infilling database.
# For the FOD, this will be fine.
# Have to use https://zenodo.org/records/20566343
infilling_file = Path("infilling_db_cmip7_scenariomip_20566343.csv")
# Messy, but basically a secondary infilling file for minor GHGs
ghg_inversion_file = Path("cmip7_ghg_inversions.csv")

PI_YEAR = 1750
HARMONISATION_YEAR = 2023

ur = openscm_units.unit_registry

infilling_db = load_cmip7_scenariomip_infilling_db(
    filepath=infilling_file,
    check_hash=False,  # TODO: update when available
)

# CMIP7 GHG inversions
cmip7_ghg_inversions = load_cmip7_scenariomip_ghg_inversions(
    filepath=ghg_inversion_file,
)
# History
historical_emissions = load_cmip7_scenariomip_historical_emissions(
    filepath=historical_emissions_file,
    check_hash=False,
)

# Use gcages naming convention.
infilling_db = update_index_levels_func(
    infilling_db,
    {
        "variable": lambda x: convert_variable_name(
            x,
            from_convention=SupportedNamingConventions.CMIP7_SCENARIOMIP,
            to_convention=SupportedNamingConventions.GCAGES,
        )
    },
    copy=False,
)
cmip7_ghg_inversions = update_index_levels_func(
    cmip7_ghg_inversions,
    {
        "variable": lambda x: convert_variable_name(
            x,
            from_convention=SupportedNamingConventions.OPENSCM_RUNNER,
            to_convention=SupportedNamingConventions.GCAGES,
        )
    },
    copy=False,
)
historical_emissions = update_index_levels_func(
    historical_emissions,
    {
        "variable": lambda x: convert_variable_name(
            x,
            from_convention=SupportedNamingConventions.CMIP7_SCENARIOMIP,
            to_convention=SupportedNamingConventions.GCAGES,
        )
    },
    copy=False,
)

assert_harmonised(
    infilling_db,
    history=historical_emissions.reset_index(
        level=[
            lvl
            for lvl in ["model", "scenario"]
            if lvl in historical_emissions.index.names
        ],
        drop=True,
    ),
    harmonisation_time=HARMONISATION_YEAR,
    history_unit_level="unit",
    ur=ur,
)

# Notes: currently this uses RMSClosest under the hood.
# That's probably not a bad decision, and avoids the OC-BC decoupling from AR6.
# We should check though and make an active, rather than passive decision.
# We likely also want to use an updated infilling DB rather than the ScenarioMIP one,
# which was just whatever scenarios we had at the time.
infiller = CMIP7ScenarioMIPInfiller(
    infilling_db=infilling_db,
    historical_emissions=historical_emissions,
    cmip7_ghg_inversions=cmip7_ghg_inversions,
    harmonisation_year=HARMONISATION_YEAR,
    pre_industrial_year=PI_YEAR,
    run_checks=True,
    ur=ur,
)

In [ ]:
complete = infiller(harmonised_global)
# complete.to_feather("complete.feather")
complete.to_parquet("sci_emissions_complete.parquet.gzip", compression="gzip")
complete

/Users/znicholls/Documents/repos-agcec/ar7-ch5-ensemble/.pixi/envs/default/lib/python3.11/site-packages/jwt/api_jwt.py:147: InsecureKeyLengthWarning: The HMAC key is 20 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/Users/znicholls/Documents/repos-agcec/ar7-ch5-ensemble/.pixi/envs/default/lib/python3.11/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
[WARNING] 18:37:34 - pint.util: Redefining 'C' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 18:37:34 - pint.util: Redefining 'N' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 18:37:34 - pint.util: Redefining 'NOX' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 18:37:34 - pint.util: Redefining 'gNOX' (<class 